In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [3]:
X_train = df_train['text']
y_train = df_train['generated']

In [4]:
X_test = df_test['text']

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [6]:
token = Tokenizer(num_words=150000, oov_token="<OOV>")

In [7]:
token.fit_on_texts(X_train)

In [8]:
X_train = token.texts_to_sequences(X_train)

In [9]:
X_test = token.texts_to_sequences(X_test)

In [10]:
from tensorflow.keras.utils import pad_sequences

In [11]:
X_train = pad_sequences(X_train, maxlen=300, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=300, padding='post', truncating='post')

In [12]:
word_counts = min(150000, len(token.word_index) + 1)

In [13]:
max_len_train = 300

In [14]:
from keras.layers import Input, Embedding, Conv1D
from keras.layers import GlobalMaxPooling1D, Dense, Dropout
from keras.layers import Bidirectional, LSTM
from keras.models import Model

input_layer = Input(shape=(max_len_train,))

x = Embedding(
    input_dim=word_counts,
    output_dim=128,
    input_length=max_len_train
)(input_layer)

x = Conv1D(128, 3, activation='relu')(x)

x = Bidirectional(LSTM(64, return_sequences=True))(x)

x = GlobalMaxPooling1D()(x)

x = Dropout(0.5)(x)

output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

C:\Users\Zhores\Desktop\MLOps\venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
from keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=10,
    callbacks=[checkpoint]
)

Epoch 1/10
9593/9593 ━━━━━━━━━━━━━━━━━━━━ 985s 103ms/step - accuracy: 0.9880 - loss: 0.0349 - val_accuracy: 0.9969 - val_loss: 0.0101
Epoch 2/10
9593/9593 ━━━━━━━━━━━━━━━━━━━━ 967s 101ms/step - accuracy: 0.9978 - loss: 0.0080 - val_accuracy: 0.9983 - val_loss: 0.0062
Epoch 3/10
9593/9593 ━━━━━━━━━━━━━━━━━━━━ 964s 101ms/step - accuracy: 0.9988 - loss: 0.0041 - val_accuracy: 0.9969 - val_loss: 0.0097
Epoch 4/10
9593/9593 ━━━━━━━━━━━━━━━━━━━━ 965s 101ms/step - accuracy: 0.9991 - loss: 0.0028 - val_accuracy: 0.9989 - val_loss: 0.0043
Epoch 5/10
9593/9593 ━━━━━━━━━━━━━━━━━━━━ 965s 101ms/step - accuracy: 0.9993 - loss: 0.0023 - val_accuracy: 0.9990 - val_loss: 0.0034
Epoch 6/10
9593/9593 ━━━━━━━━━━━━━━━━━━━━ 964s 101ms/step - accuracy: 0.9995 - loss: 0.0017 - val_accuracy: 0.9988 - val_loss: 0.0043
Epoch 7/10
9593/9593 ━━━━━━━━━━━━━━━━━━━━ 965s 101ms/step - accuracy: 0.9996 - loss: 0.0015 - val_accuracy: 0.9987 - val_loss: 0.0044
Epoch 8/10
9593/9593 ━━━━━━━━━━━━━━━━━━━━ 967s 101ms/step - ac

In [16]:
import pickle
import json

model.save("model.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(token, f)

params = {
    "max_len": 300,
    "num_words": 150000
}

with open("config.json", "w") as f:
    json.dump(params, f)

In [17]:
from keras.models import load_model
import pickle
import json

model = load_model("model.keras")

with open("tokenizer.pkl", "rb") as f:
    token = pickle.load(f)

with open("config.json") as f:
    params = json.load(f)

max_len = params["max_len"]

In [18]:
pred = model.predict(X_test, batch_size=256)

339/339 ━━━━━━━━━━━━━━━━━━━━ 35s 103ms/step


In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

text = ["This movie is amazing"]

seq = token.texts_to_sequences(text)

seq = pad_sequences(seq, maxlen=max_len)

pred = model.predict(seq)
pred_class = (pred > 0.5).astype(int)
print(pred)
print(pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
[[0.9436573]]
[[1]]


In [16]:
from keras.models import load_model
import pickle
import json

model = load_model("model.keras")

with open("tokenizer.pkl", "rb") as f:
    token = pickle.load(f)

with open("config.json") as f:
    params = json.load(f)

max_len = params["max_len"]

In [17]:
pred = model.predict(X_test, batch_size=64)

1353/1353 ━━━━━━━━━━━━━━━━━━━━ 39s 29ms/step


In [18]:
df_sub = pd.read_csv('sample_submission.csv')

In [19]:
df_sub.info()

<class 'pandas.DataFrame'>
RangeIndex: 86587 entries, 0 to 86586
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Id         86587 non-null  int64
 1   generated  86587 non-null  int64
dtypes: int64(2)
memory usage: 1.3 MB


In [20]:
df_sub['generated'] = pred

In [21]:
df_sub

,Id,generated
0,1,5.734988e-08
1,2,4.902923e-09
2,3,2.377030e-09
3,4,1.000000e+00
4,5,6.340994e-09
...,...,...
86582,86583,2.365131e-09
86583,86584,2.698474e-09
86584,86585,4.852684e-09
86585,86586,3.291981e-09


In [22]:
pred_class = (pred > 0.5).astype(int)

In [23]:
df_sub['generated'] = pred_class

In [25]:
df_sub.to_csv("submission.csv", index=False)